# load raw data from Hugging Face
https://huggingface.co/datasets/Zihan1004/FNSPID

## Partition

In [ ]:
!wget https://huggingface.co/datasets/Zihan1004/FNSPID/resolve/main/Stock_news/nasdaq_exteral_data.csv

In [ ]:
import pandas as pd

chunks = pd.read_csv(
    "../data/nasdaq_external_data.csv",
    chunksize=200000,       # SAFE size
    low_memory=False,
    encoding_errors="ignore"
)

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i}...")
    chunk.to_parquet(f"../data/raw_news/news_part_{i}.parquet")


## Select required columns

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq
import glob
import pandas as pd

files = sorted(glob.glob("../data/raw_news/news_part_*.parquet"))

# Keep only essential columns
KEEP = [
    "Date",
    "Stock_symbol",
    "Article_title",
    "Lsa_summary",
    "Url"
]

# Define dtype for consistent schema
DTYPES = {
    "Date": "string",
    "Stock_symbol": "string",
    "Article_title": "string",
    "Lsa_summary": "string",
    "Url": "string"
}

first = True
writer = None

for f in files:
    print("Appending:", f)

    df = pd.read_parquet(f)

    # Keep only the necessary columns
    df = df[KEEP]

    # Enforce consistent dtype
    for col, t in DTYPES.items():
        df[col] = df[col].astype(t)

    table = pa.Table.from_pandas(df)

    if first:
        writer = pq.ParquetWriter(
            "../data/raw_news/nasdaq_clean.parquet",
            table.schema
        )
        first = False

    writer.write_table(table)

writer.close()

print("Finished combining clean dataset!")


Appending: ../data/raw_news/news_part_0.parquet
Appending: ../data/raw_news/news_part_1.parquet
Appending: ../data/raw_news/news_part_10.parquet
Appending: ../data/raw_news/news_part_11.parquet
Appending: ../data/raw_news/news_part_12.parquet
Appending: ../data/raw_news/news_part_13.parquet
Appending: ../data/raw_news/news_part_14.parquet
Appending: ../data/raw_news/news_part_15.parquet
Appending: ../data/raw_news/news_part_16.parquet
Appending: ../data/raw_news/news_part_17.parquet
Appending: ../data/raw_news/news_part_18.parquet
Appending: ../data/raw_news/news_part_19.parquet
Appending: ../data/raw_news/news_part_2.parquet
Appending: ../data/raw_news/news_part_20.parquet
Appending: ../data/raw_news/news_part_21.parquet
Appending: ../data/raw_news/news_part_22.parquet
Appending: ../data/raw_news/news_part_23.parquet
Appending: ../data/raw_news/news_part_24.parquet
Appending: ../data/raw_news/news_part_25.parquet
Appending: ../data/raw_news/news_part_26.parquet
Appending: ../data/raw_

In [2]:
import pandas as pd

df = pd.read_parquet("../data/raw_news/nasdaq_clean.parquet", engine="pyarrow")
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15549299 entries, 0 to 15549298
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   Date           string
 1   Stock_symbol   string
 2   Article_title  string
 3   Lsa_summary    string
 4   Url            string
dtypes: string(5)
memory usage: 593.2 MB


In [3]:
df

,Date,Stock_symbol,Article_title,Lsa_summary,Url
0,2023-12-16 23:00:00 UTC,A,Interesting A Put And Call Options For August ...,Because the $125.00 strike represents an appro...,https://www.nasdaq.com/articles/interesting-a-...
1,2023-12-12 00:00:00 UTC,A,Wolfe Research Initiates Coverage of Agilent T...,"Fintel reports that on December 13, 2023, Wolf...",https://www.nasdaq.com/articles/wolfe-research...
2,2023-12-12 00:00:00 UTC,A,Agilent Technologies Reaches Analyst Target Price,"In recent trading, shares of Agilent Technolog...",https://www.nasdaq.com/articles/agilent-techno...
3,2023-12-07 00:00:00 UTC,A,Agilent (A) Enhances BioTek Cytation C10 With ...,"Per a Grand View Research report, the global m...",https://www.nasdaq.com/articles/agilent-a-enha...
4,2023-12-07 00:00:00 UTC,A,"Pre-Market Most Active for Dec 7, 2023 : SQQQ,...",ProShares UltraPro Short QQQ (SQQQ) is -0.15 a...,https://www.nasdaq.com/articles/pre-market-mos...
...,...,...,...,...,...
15549294,2023-12-08 00:00:00 UTC,SOHOO,Why RH Stock Crumbled Friday,In a comment that summed up the poor sales env...,https://www.nasdaq.com/articles/why-rh-stock-c...
15549295,2023-12-08 00:00:00 UTC,SOHOO,Merck's (MRK) Keytruda-Lynparza Combo Fails Lu...,Merck MRK announced that it will stop the phas...,https://www.nasdaq.com/articles/mercks-mrk-key...
15549296,2023-12-08 00:00:00 UTC,SOHOO,Insights Into REV Group (REVG) Q4: Wall Street...,"In its upcoming report, REV Group (REVG) is pr...",https://www.nasdaq.com/articles/insights-into-...
15549297,2023-12-08 00:00:00 UTC,SOHOO,OSIS or OLED: Which Is the Better Value Stock ...,There are plenty of strategies for discovering...,https://www.nasdaq.com/articles/osis-or-oled%3...


## Select sp500 companies

In [7]:
# load the sp500 tickers
import pandas as pd

sp500 = pd.read_csv("../data/sp500_companies.csv")
sp500_tickers = set(sp500["ticker"].str.upper())
print("S&P500 tickers loaded:", len(sp500_tickers))



S&P500 tickers loaded: 503


In [5]:
sp500

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373
...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,1524472
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,1041061
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,877212
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,1136869


In [6]:
df.columns

Index(['Date', 'Stock_symbol', 'Article_title', 'Lsa_summary', 'Url'], dtype='object')

In [8]:
df["Stock_symbol"] = df["Stock_symbol"].str.upper()

df = df[df["Stock_symbol"].isin(sp500_tickers)]
print("After S&P500 filter:", df.shape)


After S&P500 filter: (1395032, 5)


In [9]:
# find unique tickers, and count each of them
unique_tickers = df["Stock_symbol"].value_counts()
print("Unique tickers after filter:", len(unique_tickers))
unique_tickers

Unique tickers after filter: 458


Stock_symbol
GILD    12376
NVDA    11862
WFC     11301
INTC    11157
MRK     10774
        ...  
LVS        97
SCHW       38
EXE        26
LW          7
LIN         7
Name: count, Length: 458, dtype: Int64

In [11]:
# find the smallest date
df["Date"] = pd.to_datetime(df["Date"])
min_date = df["Date"].max()
print("Earliest date in filtered data:", min_date)

Earliest date in filtered data: 2024-01-09 00:00:00+00:00


/tmp/ipykernel_2707671/3821741406.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Date"] = pd.to_datetime(df["Date"])


In [12]:
df = df[(df["Date"] >= "2014-01-01") & (df["Date"] < "2024-01-01")]
print("Filtered shape:", df.shape)
print("Date range:", df["Date"].min(), df["Date"].max())


Filtered shape: (1194442, 5)
Date range: 2014-01-01 00:00:00+00:00 2023-12-31 00:00:00+00:00


In [ ]:
train_df = df[(df["Date"] >= "2014-01-01") & (df["Date"] < "2022-01-01")]
val_df   = df[(df["Date"] >= "2022-01-01") & (df["Date"] < "2023-01-01")]
test_df  = df[(df["Date"] >= "2023-01-01") & (df["Date"] < "2024-01-01")]

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)



Train: (921967, 5)
Val: (112846, 5)
Test: (159629, 5)


In [15]:
train_df.to_parquet("../data/model/train_news.parquet")
val_df.to_parquet("../data/model/val_news.parquet")
test_df.to_parquet("../data/model/test_news.parquet")

print("Saved all splits!")


Saved all splits!
